# 🩺 Medical Consultation — Real-Time Transcription Pipeline

**Scope of this notebook:** live microphone capture → Silero VAD → speech
segmentation/endpointing → Faster-Whisper ASR → transcript post-processing →
incremental transcript merge → live Gradio display.

**Out of scope (future stage):** medical information extraction, clinical
summary, diagnosis, patient JSON, LLM reasoning, speaker diarization,
database, authentication.

```
🎤 Microphone → 🌐 Gradio UI → ☁️ Colab → 16kHz Audio → Silero VAD →
Speech Buffer → Endpoint Detection → Faster-Whisper → Transcript
Processing → Live Transcript → 🌐 Gradio UI
```

Run the cells top to bottom. The last cell launches the Gradio app with a
public URL you can open in your browser.


## Cell 1 — Environment Check

Reports Python/PyTorch/CUDA/GPU/CPU so you know whether this run is on CPU or GPU before anything else happens.

In [1]:
import sys
import platform
import multiprocessing

def print_environment():
    print("Environment")
    print("-----------")
    print(f"Python: {sys.version.split()[0]}")

    try:
        import torch
        print(f"PyTorch: {torch.__version__}")
        cuda_available = torch.cuda.is_available()
        print(f"CUDA: {'Available' if cuda_available else 'Not available'}")
        if cuda_available:
            print(f"GPU: {torch.cuda.get_device_name(0)}")
            props = torch.cuda.get_device_properties(0)
            print(f"GPU Memory: {props.total_memory / (1024**3):.1f} GB")
        else:
            print("GPU: None (will run on CPU)")
    except ImportError:
        print("PyTorch: not installed yet (install in Cell 2)")
        cuda_available = False

    print(f"CPU: {platform.processor() or platform.machine()}")
    print(f"CPU cores (logical): {multiprocessing.cpu_count()}")
    print(f"Platform: {platform.platform()}")

    return cuda_available

# This will report "not installed yet" the first time you run the notebook
# top-to-bottom (before Cell 2). Re-run this cell after Cell 2 if you want
# the full torch/CUDA report.
GPU_AVAILABLE = print_environment()


Environment
-----------
Python: 3.13.15
PyTorch: 2.11.0+cu128
CUDA: Available
GPU: Tesla T4
GPU Memory: 14.6 GB
CPU: x86_64
CPU cores (logical): 2
Platform: Linux-6.6.122+-x86_64-with-glibc2.39


## Cell 2 — Install Dependencies

Only what the pipeline needs. `torch`/`torchaudio` are usually already
present in Colab; the install is safe to re-run (pip skips satisfied
requirements). Silero VAD is loaded via `torch.hub` in Cell 3, so no
separate `silero-vad` package is required — but we pin `onnxruntime` off
by default and rely on the PyTorch JIT model, which is the officially
documented real-time path.

In [2]:
%%capture install_log
# Reproducible, minimal install. Re-running this cell is safe/idempotent.
!pip install -q --upgrade \
    faster-whisper==1.0.3 \
    gradio==4.44.1 \
    numpy \
    scipy \
    soundfile \
    torch \
    torchaudio


In [3]:
# Show the tail of the install log only if something looks wrong.
try:
    install_log.show()
except Exception:
    pass
print("Dependencies installed.")

# Re-report environment now that torch is guaranteed to be importable.
GPU_AVAILABLE = print_environment()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 67.2 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Dependencies installed.
Environment
-----------
Python: 3.13.15
PyTorch: 2.11.0+cu128
CUDA: Available
GPU: Tesla T4
GPU Memory: 14.6 GB
CPU: x86_64
CPU cores (logical): 2
Platform: Linux-6.6.122+-x86_64-with-glibc2.39


## Cell 3 — Load Silero VAD

Loads the Silero VAD model via `torch.hub` (the model's documented
real-time API) and wraps it in a small reusable `SileroVAD` class:

```
audio frame → speech probability → speech / non-speech
```

All tunables live here in one place — no magic numbers scattered later.

In [ ]:
import torch
import numpy as np

# ---------------------------------------------------------------------------
# Configurable VAD parameters (single source of truth)
# ---------------------------------------------------------------------------
VAD_THRESHOLD = 0.5            # speech-probability cutoff (0-1)
MIN_SPEECH_DURATION_MS = 200   # ignore speech runs shorter than this
MIN_SILENCE_DURATION_MS = 100  # frame-level silence smoothing
PRE_SPEECH_PADDING_MS = 300    # audio kept from just before speech onset
POST_SPEECH_PADDING_MS = 300   # audio kept from just after speech offset
END_OF_SPEECH_MS = 600         # trailing silence that finalizes an utterance

torch.set_num_threads(1)  # Silero VAD recommendation for low-latency inference


class SileroVAD:
    """Thin, reusable wrapper around the Silero VAD JIT model.

    Exposes a single method: speech_prob(frame) -> float in [0, 1].
    The frame must be 16 kHz mono float32 PCM.
    """

    def __init__(self, sample_rate: int = 16000):
        self.sample_rate = sample_rate
        try:
            self.model, _utils = torch.hub.load(
                repo_or_dir="snakers4/silero-vad",
                model="silero_vad",
                force_reload=False,
                onnx=False,
                trust_repo=True,
            )
            self.model.eval()
            self._loaded = True
        except Exception as exc:  # noqa: BLE001 - surfaced to the UI later
            self._loaded = False
            self._load_error = exc

    @property
    def is_loaded(self) -> bool:
        return self._loaded

    def speech_prob(self, frame: np.ndarray) -> float:
        """Return the probability that `frame` (float32, mono, 16kHz) contains speech."""
        if not self._loaded:
            raise RuntimeError(f"Silero VAD failed to load: {getattr(self, '_load_error', 'unknown error')}")
        with torch.no_grad():
            tensor = torch.from_numpy(frame).float()
            prob = self.model(tensor, self.sample_rate).item()
        return prob

    def reset(self):
        """Reset the model's internal streaming state (call between sessions)."""
        if self._loaded and hasattr(self.model, "reset_states"):
            self.model.reset_states()


vad = SileroVAD(sample_rate=16000)
if vad.is_loaded:
    print("Silero VAD loaded.")
else:
    print(f"⚠️ Silero VAD failed to load: {vad._load_error}")


## Cell 4 — Load Faster-Whisper

Model size is configurable (`tiny` / `base` / `small`, ...). Device and
compute type are chosen automatically: CUDA + `float16` when a GPU is
available, CPU + `int8` otherwise. Loading failures are caught and
surfaced instead of crashing the notebook.

In [ ]:
from faster_whisper import WhisperModel

# ---------------------------------------------------------------------------
# Configurable ASR parameters
# ---------------------------------------------------------------------------
WHISPER_MODEL = "small"     # one of: tiny, base, small
LANGUAGE = "en"              # set to None to auto-detect language per segment
BEAM_SIZE = 1                # greedy-ish decoding, optimized for latency

WHISPER_DEVICE = "cuda" if GPU_AVAILABLE else "cpu"
WHISPER_COMPUTE_TYPE = "float16" if GPU_AVAILABLE else "int8"

whisper_model = None
whisper_load_error = None

try:
    whisper_model = WhisperModel(
        WHISPER_MODEL,
        device=WHISPER_DEVICE,
        compute_type=WHISPER_COMPUTE_TYPE,
    )
    print(f"Faster-Whisper '{WHISPER_MODEL}' loaded on {WHISPER_DEVICE} "
          f"(compute_type={WHISPER_COMPUTE_TYPE}).")
except Exception as exc:  # noqa: BLE001
    whisper_load_error = exc
    print(f"⚠️ Whisper model could not be loaded: {exc}")


## Cell 5 — Audio Configuration

Defines the common internal audio format (16 kHz mono float32) and a
normalization function that converts whatever Gradio hands us into that
format. Every later stage assumes this format.

In [ ]:
from scipy.signal import resample_poly
from math import gcd

SAMPLE_RATE = 16000
CHANNELS = 1


def normalize_audio(raw_audio: np.ndarray, orig_sr: int) -> np.ndarray:
    """Convert incoming Gradio audio into 16kHz mono float32 PCM.

    incoming Gradio audio -> normalize dtype -> mono -> resample to 16kHz -> float32
    """
    audio = np.asarray(raw_audio)

    # 1. Normalize representation: integer PCM -> float32 in [-1, 1]
    if np.issubdtype(audio.dtype, np.integer):
        max_val = float(np.iinfo(audio.dtype).max)
        audio = audio.astype(np.float32) / max_val
    else:
        audio = audio.astype(np.float32)

    # 2. Convert to mono (Gradio may hand back (n,) or (n, channels))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # 3. Resample to 16kHz if needed
    if orig_sr != SAMPLE_RATE and len(audio) > 0:
        g = gcd(SAMPLE_RATE, orig_sr)
        up, down = SAMPLE_RATE // g, orig_sr // g
        audio = resample_poly(audio, up, down).astype(np.float32)

    return np.ascontiguousarray(audio, dtype=np.float32)


print(f"Audio pipeline configured: {SAMPLE_RATE} Hz, {CHANNELS} channel(s), float32 PCM.")


## Cell 6 — Audio Framing

Splits normalized audio into small fixed-size frames (20–30 ms) before
handing them to VAD. Whisper never sees raw frames — only completed
utterances (Cell 9/14).

In [ ]:
from dataclasses import dataclass
from typing import List

FRAME_MS = 30  # 20-30ms recommended; 30ms => 480 samples @ 16kHz
FRAME_SAMPLES = int(SAMPLE_RATE * FRAME_MS / 1000)


@dataclass
class AudioFrame:
    samples: np.ndarray   # float32, length == FRAME_SAMPLES (last frame may be shorter/padded)
    timestamp: float       # seconds, start time of the frame within the session
    duration: float        # seconds


def frame_generator(audio: np.ndarray, start_timestamp: float = 0.0) -> List[AudioFrame]:
    """Slice `audio` (16kHz mono float32) into FRAME_MS frames.

    The final partial frame, if any, is zero-padded to a full frame so the
    VAD model always receives a consistent input length.
    """
    frames = []
    n = len(audio)
    step = FRAME_SAMPLES
    for i in range(0, n, step):
        chunk = audio[i:i + step]
        if len(chunk) < step:
            chunk = np.pad(chunk, (0, step - len(chunk)))
        ts = start_timestamp + i / SAMPLE_RATE
        frames.append(AudioFrame(samples=chunk, timestamp=ts, duration=FRAME_MS / 1000))
    return frames


print(f"Frame size: {FRAME_MS} ms = {FRAME_SAMPLES} samples @ {SAMPLE_RATE} Hz.")


## Cell 7 — VAD State Machine + Speech Buffer

An explicit state machine keeps context across frames instead of
reacting to each frame independently:

```
IDLE → SPEECH_DETECTED → IN_SPEECH → POSSIBLE_END → UTTERANCE_COMPLETE → IDLE
```

`SpeechBuffer` accumulates pre-speech padding + speech frames + post-speech
padding while the state machine is in `IN_SPEECH` / `POSSIBLE_END`.

In [ ]:
from enum import Enum, auto
from collections import deque
from typing import Optional


class VADState(Enum):
    IDLE = auto()
    SPEECH_DETECTED = auto()
    IN_SPEECH = auto()
    POSSIBLE_END = auto()
    UTTERANCE_COMPLETE = auto()


class SpeechBuffer:
    """Accumulates frames for a single in-progress utterance, including
    pre-speech and post-speech padding."""

    def __init__(self):
        pre_pad_frames = max(1, int(PRE_SPEECH_PADDING_MS / FRAME_MS))
        self._pre_ring: deque = deque(maxlen=pre_pad_frames)
        self._speech_frames: List[AudioFrame] = []
        self._post_pad_target = max(1, int(POST_SPEECH_PADDING_MS / FRAME_MS))
        self._post_pad_count = 0
        self.active = False

    def observe_idle_frame(self, frame: AudioFrame):
        """Feed frames seen while IDLE into the pre-speech ring buffer."""
        self._pre_ring.append(frame)

    def start(self):
        self.active = True
        self._speech_frames = list(self._pre_ring)  # seed with pre-speech padding
        self._post_pad_count = 0

    def add_speech_frame(self, frame: AudioFrame):
        self._speech_frames.append(frame)

    def add_post_padding_frame(self, frame: AudioFrame) -> bool:
        """Add a trailing (silence) frame as post-speech padding.

        Returns True once enough post-speech padding has been collected.
        """
        self._speech_frames.append(frame)
        self._post_pad_count += 1
        return self._post_pad_count >= self._post_pad_target

    def get_audio(self) -> np.ndarray:
        if not self._speech_frames:
            return np.array([], dtype=np.float32)
        return np.concatenate([f.samples for f in self._speech_frames])

    def duration_ms(self) -> float:
        return len(self._speech_frames) * FRAME_MS

    def reset(self):
        self._speech_frames = []
        self._post_pad_count = 0
        self.active = False


print("VAD states:", [s.name for s in VADState])


## Cell 8 — Endpoint Detection (Utterance Segmenter)

Ties the state machine + `SpeechBuffer` + trailing-silence endpointing
together. Feed it frame-by-frame; it returns a completed utterance's audio
once `END_OF_SPEECH_MS` of trailing silence is observed after speech.

```
silence silence speech speech speech speech silence silence silence
                                                        → utterance complete
```


In [ ]:
class UtteranceSegmenter:
    """Frame-by-frame VAD state machine producing complete speech utterances.

    process_frame(frame) -> Optional[np.ndarray]
        Returns the finalized utterance audio (float32, 16kHz) when an
        utterance completes, otherwise None.
    """

    def __init__(self, vad_model: SileroVAD):
        self.vad = vad_model
        self.state = VADState.IDLE
        self.buffer = SpeechBuffer()
        self._silence_ms_in_speech = 0
        self._speech_ms_in_run = 0

    def process_frame(self, frame: AudioFrame) -> Optional[np.ndarray]:
        prob = self.vad.speech_prob(frame.samples)
        is_speech = prob >= VAD_THRESHOLD

        if self.state == VADState.IDLE:
            self.buffer.observe_idle_frame(frame)
            if is_speech:
                self.state = VADState.SPEECH_DETECTED
                self._speech_ms_in_run = FRAME_MS
            return None

        if self.state == VADState.SPEECH_DETECTED:
            if is_speech:
                self._speech_ms_in_run += FRAME_MS
                if self._speech_ms_in_run >= MIN_SPEECH_DURATION_MS:
                    self.buffer.start()
                    for f in []:  # frames already seeded from pre-ring in start()
                        pass
                    self.buffer.add_speech_frame(frame)
                    self.state = VADState.IN_SPEECH
                    self._silence_ms_in_speech = 0
            else:
                # Too short to count as real speech onset; go back to idle.
                self.buffer.observe_idle_frame(frame)
                self.state = VADState.IDLE
            return None

        if self.state == VADState.IN_SPEECH:
            if is_speech:
                self.buffer.add_speech_frame(frame)
                self._silence_ms_in_speech = 0
            else:
                self.state = VADState.POSSIBLE_END
                self._silence_ms_in_speech = FRAME_MS
                self.buffer.add_post_padding_frame(frame)
            return None

        if self.state == VADState.POSSIBLE_END:
            if is_speech:
                # Speech resumed before endpoint threshold: back to IN_SPEECH.
                self.buffer.add_speech_frame(frame)
                self.state = VADState.IN_SPEECH
                self._silence_ms_in_speech = 0
                return None

            self._silence_ms_in_speech += FRAME_MS
            padding_done = self.buffer.add_post_padding_frame(frame)
            if self._silence_ms_in_speech >= END_OF_SPEECH_MS and padding_done:
                self.state = VADState.UTTERANCE_COMPLETE
                utterance_audio = self.buffer.get_audio()
                self.buffer.reset()
                self.state = VADState.IDLE
                if len(utterance_audio) == 0:
                    return None
                return utterance_audio
            return None

        return None

    def flush(self) -> Optional[np.ndarray]:
        """Force-finalize any in-progress utterance (used on Stop)."""
        if self.state in (VADState.IN_SPEECH, VADState.POSSIBLE_END) and self.buffer.active:
            utterance_audio = self.buffer.get_audio()
            self.buffer.reset()
            self.state = VADState.IDLE
            if len(utterance_audio) == 0:
                return None
            return utterance_audio
        return None


print("UtteranceSegmenter ready.")


## Cell 9 — Segment Preprocessing (pre-Whisper)

Lightweight, deterministic cleanup of a finalized speech segment before it
goes to Whisper: mono/16kHz guarantee, peak volume normalization, and an
optional simple high-pass filter to reduce low-frequency rumble. No LLMs,
no heavy DSP — keeping latency low is the priority.

In [ ]:
from scipy.signal import butter, sosfilt

APPLY_NOISE_SUPPRESSION = True
HIGHPASS_CUTOFF_HZ = 80  # removes low-frequency rumble/handling noise


def _highpass(audio: np.ndarray, cutoff_hz: int = HIGHPASS_CUTOFF_HZ, sr: int = SAMPLE_RATE) -> np.ndarray:
    if len(audio) < 32:
        return audio
    sos = butter(2, cutoff_hz, btype="highpass", fs=sr, output="sos")
    return sosfilt(sos, audio).astype(np.float32)


def preprocess_segment(audio: np.ndarray, orig_sr: int = SAMPLE_RATE) -> np.ndarray:
    """Speech segment -> mono/16kHz -> volume normalization -> (optional) noise suppression -> Whisper input."""
    audio = normalize_audio(audio, orig_sr)

    if len(audio) == 0:
        return audio

    if APPLY_NOISE_SUPPRESSION:
        audio = _highpass(audio)

    peak = np.max(np.abs(audio))
    if peak > 1e-6:
        audio = (audio / peak) * 0.95  # normalize to ~95% full scale, avoid clipping

    return audio.astype(np.float32)


print("Segment preprocessing ready.")


## Cell 10 — Whisper Transcription Function

One clean abstraction: `transcribe_segment(audio_segment) -> dict`.
Uses greedy/low-beam decoding for latency, and a fixed `LANGUAGE` when
known to avoid repeated language detection. Failures are caught so one
bad segment never kills the session.

In [ ]:
import time


def transcribe_segment(audio_segment: np.ndarray) -> dict:
    """Transcribe a 16kHz mono float32 speech segment with Faster-Whisper.

    Returns: {"text": str, "language": str, "duration": float, "error": Optional[str]}
    """
    if whisper_model is None:
        return {"text": "", "language": None, "duration": 0.0,
                "error": "Whisper model could not be loaded."}

    if audio_segment is None or len(audio_segment) == 0:
        return {"text": "", "language": None, "duration": 0.0, "error": None}

    try:
        segments, info = whisper_model.transcribe(
            audio_segment,
            language=LANGUAGE,
            beam_size=BEAM_SIZE,
            vad_filter=False,     # we already did VAD/endpointing ourselves
            condition_on_previous_text=False,
        )
        text = "".join(seg.text for seg in segments).strip()
        return {
            "text": text,
            "language": getattr(info, "language", LANGUAGE),
            "duration": getattr(info, "duration", len(audio_segment) / SAMPLE_RATE),
            "error": None,
        }
    except Exception as exc:  # noqa: BLE001
        return {"text": "", "language": None, "duration": 0.0, "error": str(exc)}


print("transcribe_segment() ready.")


## Cell 11 — Transcript Post-Processing

Purely mechanical text cleanup — no medical reasoning. Whitespace,
capitalization, punctuation, and simple repeated-word/phrase cleanup
only.

In [ ]:
import re


def clean_transcript_text(text: str) -> str:
    """raw Whisper text -> whitespace cleanup -> capitalization ->
    punctuation normalization -> duplicate-fragment cleanup -> final text."""
    if not text:
        return ""

    # Whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return ""

    # Collapse immediately-repeated words ("the the" -> "the")
    text = re.sub(r"\b(\w+)( \1\b)+", r"\1", text, flags=re.IGNORECASE)

    # Capitalize first letter of the segment
    text = text[0].upper() + text[1:] if len(text) > 1 else text.upper()

    # Ensure terminal punctuation
    if text and text[-1] not in ".!?":
        text += "."

    return text


# Quick self-check (not a medical inference — just mechanical cleanup)
assert clean_transcript_text("i have chest pain for two days") == "I have chest pain for two days."
print("clean_transcript_text() ready.")


## Cell 12 — Incremental Transcript Merger

Appends each completed, cleaned segment to a running session transcript,
using a simple deterministic word-overlap check at segment boundaries to
avoid duplicated fragments (e.g. "chest pain" followed by "chest pain for
two days" should not double up).

In [ ]:
MAX_BOUNDARY_OVERLAP_WORDS = 6


def _dedupe_boundary(previous: str, new: str) -> str:
    """If `new` starts with a word sequence that overlaps the tail of
    `previous`, strip the overlapping prefix from `new`."""
    prev_words = previous.lower().rstrip(".!?").split()
    new_words_raw = new.split()
    new_words_lower = new.lower().split()

    max_k = min(MAX_BOUNDARY_OVERLAP_WORDS, len(prev_words), len(new_words_lower))
    for k in range(max_k, 0, -1):
        if prev_words[-k:] == new_words_lower[:k]:
            remainder = new_words_raw[k:]
            return " ".join(remainder)
    return new


class TranscriptMerger:
    """Maintains the session transcript as an ordered list of cleaned segments."""

    def __init__(self):
        self.transcript_segments: List[str] = []

    def add_segment(self, cleaned_text: str) -> Optional[str]:
        """Add a cleaned segment, deduplicating boundary overlap with the
        previous segment. Returns the segment actually appended (or None
        if it was entirely a duplicate)."""
        if not cleaned_text:
            return None

        if self.transcript_segments:
            remainder = _dedupe_boundary(self.transcript_segments[-1], cleaned_text)
            if not remainder:
                return None
            remainder = remainder[0].upper() + remainder[1:] if len(remainder) > 1 else remainder.upper()
            if remainder[-1] not in ".!?":
                remainder += "."
            cleaned_text = remainder

        self.transcript_segments.append(cleaned_text)
        return cleaned_text

    def get_transcript(self) -> str:
        return "\n\n".join(self.transcript_segments)

    def reset(self):
        self.transcript_segments = []


print("TranscriptMerger ready.")


## Cell 13 — Pipeline Integration

Wires everything into one `ConsultationPipeline` object that owns
per-session state (no shared globals across sessions — each Gradio
session gets its own instance via `gr.State`). Also handles error
isolation (Cell 18 requirements) and per-segment latency logging
(Cell 19 requirements).

In [ ]:
import traceback


class ConsultationPipeline:
    """Owns one live-transcription session end-to-end:
    audio chunk -> frames -> VAD segmenter -> utterance -> preprocess ->
    Whisper -> post-process -> merge -> transcript.
    """

    def __init__(self):
        self.vad = SileroVAD(sample_rate=SAMPLE_RATE)  # fresh streaming state per session
        self.segmenter = UtteranceSegmenter(self.vad)
        self.merger = TranscriptMerger()

        self.is_recording = False
        self.segments_processed = 0
        self.total_inference_time = 0.0
        self.last_error: Optional[str] = None
        self.status = "Ready"

        self._session_start = time.time()

    # -- state dict expected by the assignment spec (Section 17) -----------
    def session_state(self) -> dict:
        return {
            "is_recording": self.is_recording,
            "transcript_segments": list(self.merger.transcript_segments),
            "final_transcript": self.merger.get_transcript(),
        }

    def _handle_utterance(self, utterance_audio: np.ndarray) -> Optional[str]:
        """Run one utterance through preprocess -> Whisper -> post-process -> merge."""
        if utterance_audio is None or len(utterance_audio) == 0:
            return None  # Empty speech: do nothing (Section 18)

        t0 = time.time()
        try:
            audio_duration = len(utterance_audio) / SAMPLE_RATE
            processed = preprocess_segment(utterance_audio, SAMPLE_RATE)

            t_whisper_start = time.time()
            result = transcribe_segment(processed)
            whisper_time = time.time() - t_whisper_start

            if result.get("error"):
                self.last_error = f"Whisper failed on one segment: {result['error']}"
                print(f"[SEGMENT ERROR] {self.last_error}")
                return None  # continue listening; don't kill the session

            cleaned = clean_transcript_text(result["text"])
            if not cleaned:
                return None

            appended = self.merger.add_segment(cleaned)

            self.segments_processed += 1
            total_time = time.time() - t0
            self.total_inference_time += total_time
            print(
                f"[SEGMENT] Audio duration: {audio_duration:.2f}s | "
                f"Whisper: {whisper_time:.2f}s | Total: {total_time:.2f}s"
            )
            return appended
        except Exception:
            self.last_error = "Unexpected error while processing a speech segment."
            print("[SEGMENT ERROR]\n" + traceback.format_exc())
            return None

    def process_audio_chunk(self, chunk: np.ndarray, sr: int) -> str:
        """Feed one Gradio audio chunk through VAD/segmentation. Returns
        current status string. Updates self.merger as utterances complete."""
        if not self.is_recording:
            return self.status

        try:
            audio = normalize_audio(chunk, sr)
        except Exception:
            self.status = "Error"
            self.last_error = "Microphone unavailable or permission denied."
            return self.status

        if len(audio) == 0:
            return self.status

        frames = frame_generator(audio)
        for frame in frames:
            try:
                utterance = self.segmenter.process_frame(frame)
            except Exception:
                self.status = "Error"
                self.last_error = "VAD failure on this chunk; continuing."
                print("[VAD ERROR]\n" + traceback.format_exc())
                continue

            if self.segmenter.state == VADState.IN_SPEECH:
                self.status = "Speech detected"
            elif self.segmenter.state == VADState.POSSIBLE_END:
                self.status = "Processing speech"

            if utterance is not None:
                self.status = "Transcribing"
                appended = self._handle_utterance(utterance)
                self.status = "Transcript updated" if appended else "Listening"

        if self.status not in ("Transcribing", "Speech detected", "Processing speech", "Transcript updated"):
            self.status = "Listening"

        return self.status

    def start(self):
        self.is_recording = True
        self.status = "Listening"
        self.vad.reset()

    def stop(self) -> str:
        """Flush any buffered speech, transcribe it, merge, and finalize."""
        self.is_recording = False
        final_utterance = self.segmenter.flush()
        if final_utterance is not None:
            self._handle_utterance(final_utterance)
        self.status = "Stopped"
        return self.merger.get_transcript()

    def debug_info(self) -> str:
        avg = (self.total_inference_time / self.segments_processed) if self.segments_processed else 0.0
        device = WHISPER_DEVICE.upper()
        lines = [
            f"Model: faster-whisper {WHISPER_MODEL}",
            f"Device: {device}",
            f"Sample rate: {SAMPLE_RATE}",
            f"Channels: {CHANNELS}",
            f"VAD threshold: {VAD_THRESHOLD}",
            f"Endpoint timeout: {END_OF_SPEECH_MS} ms",
            f"Current state: {self.segmenter.state.name}",
            f"Segments processed: {self.segments_processed}",
            f"Average inference time: {avg:.2f}s",
        ]
        if self.last_error:
            lines.append(f"Last error: {self.last_error}")
        return "\n".join(lines)


print("ConsultationPipeline ready.")


## Cell 14 — Gradio UI

Streaming microphone input, explicit Start/Stop controls, live transcript
display, and a status line + collapsible debug panel. Each browser
session gets its own `ConsultationPipeline` via `gr.State` — no shared
global mutable state.

We check the installed Gradio version and use its streaming `Audio`
component (`streaming=True`, `stream_every`), which is the supported
mechanism for near-real-time chunked microphone audio in current Gradio.
If your installed version differs materially, adjust the `gr.Audio(...)`
kwargs below accordingly (documented inline).

In [ ]:
import gradio as gr

print(f"Gradio version: {gr.__version__}")

STATUS_ICONS = {
    "Ready": "⚪",
    "Listening": "🟢",
    "Speech detected": "🟡",
    "Processing speech": "🟡",
    "Transcribing": "🔵",
    "Transcript updated": "🟢",
    "Stopped": "⚪",
    "Error": "🔴",
}


def format_status(status: str) -> str:
    icon = STATUS_ICONS.get(status, "⚪")
    return f"Status: {icon} {status}"


def on_start(pipeline: Optional[ConsultationPipeline]):
    pipeline = ConsultationPipeline()  # fresh state each time Start is pressed
    pipeline.start()
    return pipeline, format_status(pipeline.status), pipeline.merger.get_transcript(), pipeline.debug_info()


def on_stream(chunk, pipeline: Optional[ConsultationPipeline]):
    if pipeline is None or not pipeline.is_recording:
        # Ignore audio events that arrive before Start / after Stop.
        return gr.skip(), gr.skip(), gr.skip()

    if chunk is None:
        return format_status(pipeline.status), pipeline.merger.get_transcript(), pipeline.debug_info()

    sr, data = chunk
    pipeline.process_audio_chunk(np.asarray(data), sr)
    return format_status(pipeline.status), pipeline.merger.get_transcript(), pipeline.debug_info()


def on_stop(pipeline: Optional[ConsultationPipeline]):
    if pipeline is None:
        return format_status("Stopped"), "", ""
    final_transcript = pipeline.stop()
    return format_status(pipeline.status), final_transcript, pipeline.debug_info()


with gr.Blocks(title="Medical Live Transcription") as demo:
    gr.Markdown("# 🩺 Medical Live Transcription")
    gr.Markdown(
        "Speech-to-text prototype: microphone → VAD → Faster-Whisper → live transcript. "
        "**Medical analysis is a future stage and is not performed here.**"
    )

    pipeline_state = gr.State(value=None)

    with gr.Row():
        mic = gr.Audio(
            sources=["microphone"],
            streaming=True,
            type="numpy",
            label="🎤 Microphone",
            show_label=True,
        )

    with gr.Row():
        start_btn = gr.Button("▶ Start Recording", variant="primary")
        stop_btn = gr.Button("⏹ Stop Recording")

    status_box = gr.Markdown(format_status("Ready"))

    gr.Markdown("### LIVE TRANSCRIPT")
    transcript_box = gr.Textbox(
        value="",
        lines=10,
        interactive=False,
        show_label=False,
        placeholder="Transcript will appear here as you speak...",
    )

    with gr.Accordion("Processing pipeline / Debug", open=False):
        gr.Markdown("`VAD → Speech → Whisper → Transcript`")
        debug_box = gr.Textbox(value="", lines=9, interactive=False, show_label=False)

    start_btn.click(
        fn=on_start,
        inputs=[pipeline_state],
        outputs=[pipeline_state, status_box, transcript_box, debug_box],
    )

    # `stream_every` throttles how often the browser pushes chunks (seconds).
    mic.stream(
        fn=on_stream,
        inputs=[mic, pipeline_state],
        outputs=[status_box, transcript_box, debug_box],
        stream_every=0.5,
    )

    stop_btn.click(
        fn=on_stop,
        inputs=[pipeline_state],
        outputs=[status_box, transcript_box, debug_box],
    )

print("Gradio Blocks app defined.")


## Cell 15 — Launch Gradio

Launches with `share=True` so Colab exposes a public URL you can open in
your browser. Grant microphone permission, click **Start Recording**, and
speak — the transcript updates live. Click **Stop Recording** to flush
and finalize.

In [ ]:
demo.queue()  # keeps Whisper inference from blocking the UI thread
demo.launch(share=True, debug=True)


## Cell 16 — Manual Acceptance Test Checklist

Run through these against the live app above:

1. **Silence** — say nothing for a few seconds. Status should stay
   `Listening`; no Whisper call happens (check the printed logs — no
   `[SEGMENT]` line); transcript unchanged.
2. **Single utterance** — say *"I have been having chest pain for two
   days."* Expect that sentence to appear shortly after you stop talking.
3. **Multiple utterances** — say three short sentences with pauses
   between them. Expect each to appear as its own line, incrementally.
4. **Silence between sentences** — confirm the pauses don't trigger extra
   Whisper calls or garbage text.
5. **Stop** — click **Stop Recording** mid-sentence or right after
   speaking; confirm the buffered tail is transcribed and appended before
   the session is marked `Stopped`.

The finalized text is available programmatically as:

```python
pipeline_state.value.merger.get_transcript()   # via the gr.State in the running demo
```

or, for a pipeline instance you hold directly:

```python
final_transcript = pipeline.stop()
```

This `final_transcript` string is the only artifact the future medical-LLM
analysis stage needs — nothing else in this notebook is required by that
stage.

## Reference — Assignment Alignment

| Assignment Requirement | Implementation |
|---|---|
| Live microphone audio | Gradio `Audio(streaming=True, sources=["microphone"])` |
| Voice Activity Detection | Silero VAD (Cell 3) |
| Silence/background filtering | `UtteranceSegmenter` state machine before Whisper (Cell 8) |
| ASR | Faster-Whisper, local inference (Cell 10) |
| Live transcript | Incremental Gradio `Textbox` updates via `mic.stream(...)` (Cell 14) |
| Python backend/AI integration | Google Colab Python runtime |
| Frontend | Gradio UI |
| API integration | Gradio browser ↔ Colab runtime over the streaming event |

**Explicitly out of scope here:** medical information extraction, patient
JSON, diagnosis, clinical reasoning/summary, LLM/RAG, database,
authentication, speaker diarization, cloud Whisper API. These belong to
the next stage, which can consume `final_transcript` from this
notebook.